# R2-Dreamer: JAX vs PyTorch Reimplementation Parity

**Goal**: Verify that the JAX reimplementation of R2-Dreamer produces equivalent results to the original PyTorch code.

## Part A: Component-Level Numerical Equivalence
Weight transfer tests — load PyTorch weights into JAX and compare forward-pass outputs.

## Part B: Training Dynamics Equivalence  
100k-step Crafter training with identical data/batch order — loss curves should track closely.

In [ ]:
import os, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
PARITY_DIR = os.path.join(ROOT, 'output', 'parity')
os.makedirs(PARITY_DIR, exist_ok=True)

## Part A: Component-Level Equivalence (Weight Transfer Tests)

Run the cross-framework pytest suite and display results.

In [ ]:
# Run cross-framework tests and capture results
result = subprocess.run(
    ['python', '-m', 'pytest', 'modules/r2dreamer/tests/test_cross_framework.py',
     '-v', '--tb=no', '-q'],
    capture_output=True, text=True, cwd=ROOT, timeout=600,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "none")

In [ ]:
# Parse pytest output into a summary table
lines = result.stdout.strip().split('\n')
test_results = []
for line in lines:
    if 'PASSED' in line or 'FAILED' in line:
        parts = line.split('::')
        if len(parts) >= 2:
            class_method = parts[-1].split(' ')[0]
            status = 'PASSED' if 'PASSED' in line else 'FAILED'
            # Extract component name from class
            class_name = parts[-2].split('::')[-1] if len(parts) >= 3 else parts[-1].split('::')[0]
            test_results.append({
                'Component': class_name.replace('Test', ''),
                'Test': class_method,
                'Status': status,
                'Tolerance': '1e-4' if 'Composed' not in class_name else '2e-3',
            })

if test_results:
    df_tests = pd.DataFrame(test_results)
    # Color formatting
    def color_status(val):
        if val == 'PASSED':
            return 'background-color: #c6efce; color: #006100'
        return 'background-color: #ffc7ce; color: #9c0006'
    
    styled = df_tests.style.map(color_status, subset=['Status'])
    display(styled)
    
    n_pass = sum(1 for r in test_results if r['Status'] == 'PASSED')
    n_total = len(test_results)
    print(f'\nResult: {n_pass}/{n_total} tests passed')
else:
    print("Could not parse test results. Raw output:")
    print(result.stdout)

## Part B: Training Dynamics (100k Crafter Steps)

Load per-step metrics from both JAX and PyTorch training runs on identical Crafter data.

In [ ]:
# Load metrics
jax_path = os.path.join(PARITY_DIR, 'jax_metrics.json')
pt_path = os.path.join(PARITY_DIR, 'pytorch_metrics.json')

assert os.path.exists(jax_path), f"JAX metrics not found at {jax_path}"
assert os.path.exists(pt_path), f"PyTorch metrics not found at {pt_path}"

with open(jax_path) as f:
    jax_data = json.load(f)
with open(pt_path) as f:
    pt_data = json.load(f)

df_jax = pd.DataFrame(jax_data)
df_pt = pd.DataFrame(pt_data)

print(f"JAX: {len(df_jax)} rows, steps {df_jax['step'].min()}-{df_jax['step'].max()}")
print(f"PyTorch: {len(df_pt)} rows, steps {df_pt['step'].min()}-{df_pt['step'].max()}")

### Loss Curves: JAX vs PyTorch Overlaid

In [ ]:
def smooth(vals, window=50):
    if len(vals) < window:
        return vals, np.arange(len(vals))
    s = np.convolve(vals, np.ones(window)/window, mode='valid')
    return s, np.arange(window - 1, len(vals))

# Map JAX metric names to PyTorch equivalents
LOSS_MAP = {
    'total_loss': ('total_loss', 'opt/loss'),
    'loss/dyn': ('loss/dyn', 'loss/dyn'),
    'loss/rep': ('loss/rep', 'loss/rep'),
    'loss/barlow': ('loss/barlow', 'loss/barlow'),
    'loss/rew': ('loss/rew', 'loss/rew'),
    'loss/con': ('loss/con', 'loss/con'),
    'loss/policy': ('loss/policy', 'loss/policy'),
    'loss/value': ('loss/value', 'loss/value'),
    'loss/repval': ('loss/repval', 'loss/repval'),
}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
colors = {'JAX': '#2196F3', 'PyTorch': '#FF9800'}

for idx, (title, (jax_key, pt_key)) in enumerate(LOSS_MAP.items()):
    ax = axes[idx]
    
    if jax_key in df_jax.columns:
        vals = df_jax[jax_key].values
        s, x = smooth(vals)
        steps = df_jax['step'].values
        ax.plot(steps[x], s, label='JAX', color=colors['JAX'], alpha=0.8, linewidth=1.5)
    
    if pt_key in df_pt.columns:
        vals = df_pt[pt_key].values
        s, x = smooth(vals)
        steps = df_pt['step'].values
        ax.plot(steps[x], s, label='PyTorch', color=colors['PyTorch'], alpha=0.8, linewidth=1.5)
    
    ax.set_title(title.replace('loss/', '').replace('_', ' ').title())
    ax.set_xlabel('Step')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(PARITY_DIR, 'loss_curves_overlaid.png'), bbox_inches='tight', dpi=200)
plt.show()

### Loss Difference: |JAX - PyTorch| Over Time

This is the most diagnostic plot. If the implementations are equivalent, the absolute difference should stay small and not grow over time.

In [ ]:
# Align steps (both logged every 100 steps)
common_steps = np.intersect1d(df_jax['step'].values, df_pt['step'].values)
jax_aligned = df_jax[df_jax['step'].isin(common_steps)].set_index('step').sort_index()
pt_aligned = df_pt[df_pt['step'].isin(common_steps)].set_index('step').sort_index()

DIFF_LOSSES = [
    ('total_loss', 'opt/loss', 'Total Loss'),
    ('loss/dyn', 'loss/dyn', 'KL (dyn)'),
    ('loss/rep', 'loss/rep', 'KL (rep)'),
    ('loss/barlow', 'loss/barlow', 'Barlow Twins'),
    ('loss/rew', 'loss/rew', 'Reward'),
    ('loss/policy', 'loss/policy', 'Policy'),
    ('loss/value', 'loss/value', 'Value'),
    ('loss/repval', 'loss/repval', 'Repval'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, (jax_key, pt_key, title) in enumerate(DIFF_LOSSES):
    ax = axes[idx]
    if jax_key in jax_aligned.columns and pt_key in pt_aligned.columns:
        diff = np.abs(jax_aligned[jax_key].values - pt_aligned[pt_key].values)
        s, x = smooth(diff, window=20)
        steps = common_steps[x]
        ax.plot(steps, s, color='#E91E63', linewidth=1.2)
        ax.fill_between(steps, 0, s, alpha=0.15, color='#E91E63')
        ax.set_title(f'|diff| {title}')
        ax.set_xlabel('Step')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

plt.tight_layout()
fig.savefig(os.path.join(PARITY_DIR, 'loss_difference.png'), bbox_inches='tight', dpi=200)
plt.show()

### Relative Difference Summary Table

In [ ]:
# Summary statistics
rows = []
for jax_key, pt_key, title in DIFF_LOSSES:
    if jax_key in jax_aligned.columns and pt_key in pt_aligned.columns:
        jax_vals = jax_aligned[jax_key].values
        pt_vals = pt_aligned[pt_key].values
        abs_diff = np.abs(jax_vals - pt_vals)
        
        # Relative diff (avoid div by zero)
        denom = np.maximum(np.abs(pt_vals), 1e-8)
        rel_diff = abs_diff / denom
        
        rows.append({
            'Loss': title,
            'JAX mean': f'{np.nanmean(jax_vals):.4f}',
            'PT mean': f'{np.nanmean(pt_vals):.4f}',
            'Mean |diff|': f'{np.nanmean(abs_diff):.4f}',
            'Max |diff|': f'{np.nanmax(abs_diff):.4f}',
            'Mean rel diff': f'{np.nanmean(rel_diff)*100:.2f}%',
            'NaN (JAX)': int(np.isnan(jax_vals).sum()),
            'NaN (PT)': int(np.isnan(pt_vals).sum()),
        })

df_summary = pd.DataFrame(rows)
display(df_summary)

### NaN Skip Rate (JAX)

The JAX implementation now includes a NaN-skip guard. This plot shows how often it triggers.

In [ ]:
if 'nan_skipped' in df_jax.columns:
    nan_rate = df_jax['nan_skipped'].mean() * 100
    nan_total = df_jax['nan_skipped'].sum()
    print(f"NaN skip rate: {nan_rate:.2f}% ({int(nan_total)}/{len(df_jax)} steps)")
    
    if nan_total > 0:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.scatter(df_jax['step'].values, df_jax['nan_skipped'].values,
                   s=2, alpha=0.5, color='red')
        ax.set_xlabel('Step')
        ax.set_ylabel('NaN skipped')
        ax.set_title('NaN-Guard Activations (JAX)')
        plt.tight_layout()
        plt.show()
    else:
        print("No NaN-guard activations during training!")
else:
    print("nan_skipped metric not found")

### Training Speed Comparison

In [ ]:
if 'step_ms' in df_jax.columns and 'step_ms' in df_pt.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    
    jax_ms = df_jax['step_ms'].values
    pt_ms = df_pt['step_ms'].values
    
    ax.bar(['JAX (Flax)', 'PyTorch'], 
           [np.median(jax_ms), np.median(pt_ms)],
           yerr=[np.std(jax_ms), np.std(pt_ms)],
           color=['#2196F3', '#FF9800'], capsize=5, edgecolor='black', linewidth=0.5)
    
    for i, (m, s) in enumerate([(np.median(jax_ms), np.std(jax_ms)),
                                  (np.median(pt_ms), np.std(pt_ms))]):
        ax.text(i, m + s + 5, f'{m:.1f} ms', ha='center', fontweight='bold')
    
    speedup = np.median(pt_ms) / np.median(jax_ms)
    ax.set_ylabel('Step time (ms)')
    ax.set_title(f'Training Step Time (JAX is {speedup:.1f}x faster)')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(os.path.join(PARITY_DIR, 'speed_comparison.png'), bbox_inches='tight', dpi=200)
    plt.show()

## Summary

### Bugs Found and Fixed During Parity Testing

| # | Bug | Severity | Discovery Method |
|---|---|---|---|
| 1 | Missing LR warmup (1000 steps) | HIGH | Code review |
| 2 | No NaN-skip guard (PyTorch has GradScaler) | HIGH | Code review |
| 3 | Imagination weight cumprod formula differed | HIGH | Code review |
| 4 | MLP heads missing RMSNorm between layers | HIGH | Weight transfer test |
| 5 | Encoder flatten order NHWC vs NCHW | HIGH | Weight transfer test |
| 6 | RSSM observe action off-by-one | HIGH | Weight transfer test |
| 7 | Barlow Twins std() ddof=0 vs ddof=1 | MEDIUM | Weight transfer test |

### Conclusion

The weight transfer tests (Part A) verify that each component produces numerically equivalent outputs when given the same weights and inputs. The training dynamics comparison (Part B) shows whether the full training loop produces similar loss trajectories when trained from scratch on identical data.